In [1]:
# !pip install reportlab
# !pip install xlsxwriter
# !pip install pyproj

<div class="alert alert-info">
    Im ersten Abschnitt werden Inputs und Deklarationen festgelegt, sowie In- und Output-Pfade gesetzt, um diese für spätere Verwendungszwecke erneut verwenden zu können. <br><br>
    Ebenfalls wird das anzustrebende Datenschema aufgezeigt und in einer Liste gespeichert, um dieses jederzeit abzurufen und anpassen zu können
</div>

In [2]:
import pandas as pd
import os
import numpy as np
import folium

from folium.plugins import FastMarkerCluster
from folium.plugins import MarkerCluster
from pathlib import Path
from datetime import datetime
from pyproj import Transformer

from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors

In [3]:
# -------------------------- Basisverzeichnis ----------------------------
base_dir = Path.cwd()

# --------------- Eingabe- und Ausgabeordner definieren ------------------
input_path = base_dir / "Gesammelte_Datenbanken"
output_path = base_dir / "Angepasste_Datenbanken"
data_scheme_path = base_dir / "globales_Datenschema"

print(f"Input-Pfad:  {input_path}")
print(f"Output-Pfad: {output_path}")
print(f"Schema-Pfad: {data_scheme_path}")

Input-Pfad:  C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Gesammelte_Datenbanken
Output-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken
Schema-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\globales_Datenschema


<div class="alert alert-info">
    <b>Zuerst wird eine Liste angelegt mit</b><br>
    - Zielparameter mit Einheit<br>
    - Zielparameter ohne Einheit<br>
    - Einheit des Zielparameters
</div>

In [4]:
# ----------------------- Folgende Attribute werden für die Bachelorarbeit benötigt ------------------
ATTRIBUTES = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L", 
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L", # umol
    "NH4_in_umol/L", # umol
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L", # umol
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L", # umol
    "HS_in_mmol/L",
    "Database_number",
]

# mmol = milli-Mol
# umol = Mikro-Mol


# ---------------------------------- 2. Spalte mit Entfernung aller Suffixe ------------------------------------
SUFFIXES = {
    "_in_mmol/L": "mmol/L",
    "_in_umol/L": "µmol/L",
    "_in_mV": "mV",
    "_in_m": "m",
    "_in_c": "°C",
    "_in_uS/cm": "µS/cm"
}

simplified, units = [], []

for attr in ATTRIBUTES:
    match = next(((s, u) for s, u in SUFFIXES.items() if s in attr), None)
    if match:
        s, u = match
        simplified.append(attr.replace(s, ""))
        units.append(u)
    else:
        simplified.append(attr)
        units.append("")
    
# ---------------------------------- Zusammengefügtes Array mit zwei Spalten -----------------------------------
target_structure = pd.DataFrame({
    "Original": ATTRIBUTES,
    "Simplified": simplified,
    "Unit": units
})


# --------------------------------------------- Anzeigen -------------------------------------------------------
display(target_structure)

,Original,Simplified,Unit
0,location,location,
1,well_or_spring_name,well_or_spring_name,
2,WGS84_latitude,WGS84_latitude,
3,WGS84_longitude,WGS84_longitude,
4,depth_bgl_in_m,depth_bgl,m
5,rock_type,rock_type,
6,stratigraphic_period,stratigraphic_period,
7,temperature_in_c,temperature,°C
8,electrical_conductivity_25c_in_uS/cm,electrical_conductivity_25c,µS/cm
9,pH,pH,


<div class="alert alert-info">
    <b>Diese Liste wird ergänzt um</b><br>
    - eine Spalte mit Umrechnungsfaktoren von mg/L in mmol/L oder umol/L<br><br>
    Hierzu werden die Umrechnungsfaktoren
</div>

In [5]:
# ---------------------- Molmassen (g/mol) nach NIST ----------------------
MOLAR_MASS = {
    "O2": 31.9988,
    "Na": 22.989769,
    "Mg": 24.305,
    "Ca": 40.078,
    "Cl": 35.45,
    "SO4": 96.06,
    "HCO3": 61.0168,
    "Li": 6.94,
    "K": 39.0983,
    "Sr": 87.62,
    "NH4": 18.038,
    "Fe": 55.845,
    "Mn": 54.938,
    "F": 18.998,
    "NO3": 62.0049,
    "H2SiO3": 78.11,
    "HS": 33.07
}


# ---------------------- Umrechnungsfaktoren: mg/L -> mmol/L bzw. µmol/L ----------------------

# ---------------------------------- Molmassen als Spalte mappen ------------------------------
target_structure["Molar_Mass_g_per_mol"] = target_structure["Simplified"].map(MOLAR_MASS)


# --------------- Maske: nur Zeilen mit Chemie-Ziel-Einheit und bekannter Molmasse ------------
mask = target_structure["Unit"].isin(["mmol/L", "µmol/L"]) & target_structure["Molar_Mass_g_per_mol"].notna()


# -------------------- Faktor berechnen (mg/L -> mmol/L bzw. µmol/L) --------------------------
# ---------- mg/L / (g/mol) * (1000 mg / 1 g) = mmol/L
# ---------- mg/L / (g/mol) * (1_000_000 mg / 1 g) = µmol/L

target_structure["Conversion_Factor_mg_per_L_to_target"] = np.where(
    mask & target_structure["Unit"].eq("mmol/L"),
    1000 / target_structure["Molar_Mass_g_per_mol"],
    np.where(
        mask & target_structure["Unit"].eq("µmol/L"),
        1_000_000 / target_structure["Molar_Mass_g_per_mol"],
        np.nan
    )
)

# ------------------------------- Ergebnis darstellen -----------------------------------------
display(
    target_structure[["Original", "Simplified", "Unit", "Molar_Mass_g_per_mol", "Conversion_Factor_mg_per_L_to_target"]]
)

,Original,Simplified,Unit,Molar_Mass_g_per_mol,Conversion_Factor_mg_per_L_to_target
0,location,location,,NaN,NaN
1,well_or_spring_name,well_or_spring_name,,NaN,NaN
2,WGS84_latitude,WGS84_latitude,,NaN,NaN
3,WGS84_longitude,WGS84_longitude,,NaN,NaN
4,depth_bgl_in_m,depth_bgl,m,NaN,NaN
5,rock_type,rock_type,,NaN,NaN
6,stratigraphic_period,stratigraphic_period,,NaN,NaN
7,temperature_in_c,temperature,°C,NaN,NaN
8,electrical_conductivity_25c_in_uS/cm,electrical_conductivity_25c,µS/cm,NaN,NaN
9,pH,pH,,NaN,NaN


<div class="alert alert-info">
    <b>Erstellen einer .pdf</b><br>
    Diese .pdf dient dazu bei händischer Auswertung der Datenbanken zu vergleichen, welche Attribute in den jeweiligen Dantesätzen vorkommen.
</div>

In [6]:
# ----------------------------- Pfad zu globalem Datenschema -----------------------------
pdf_path = Path(data_scheme_path) / "globales_Datenschema.pdf"
pdf_path.parent.mkdir(parents=True, exist_ok=True)


# --------------------------- Daten für ReportLab vorbereiten ------------------------------
data = [target_structure.columns.tolist()] + target_structure.astype(str).values.tolist()


# ---------------------------------- .pdf erstellen ----------------------------------------
doc = SimpleDocTemplate(str(pdf_path), pagesize=A4)
tbl = Table(data, repeatRows=1)
tbl.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
    ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
]))


# ---------------------------------- .pdf speichern ----------------------------------------
doc.build([tbl])
print(".pdf Pfad:", pdf_path)

.pdf Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\globales_Datenschema\globales_Datenschema.pdf


<div class="alert alert-danger">
    <h1><b>FUNKTIONEN</b></h1>
</div>

In [7]:
# -------------------------------- Hilfsfunktionen --------------------------------
def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace("__+", "_", regex=True)
    )
    return df

In [8]:
# ----------------------------- Mappings, die tatsächlich im Datafram enthalten sind -----------------------------
def _available_mapping(df: pd.DataFrame, mapping: dict) -> dict:
    return {src: tgt for src, tgt in mapping.items() if src in df.columns}

In [9]:
# ---------------------------------- Einzelne Tabellenblätter aus Excel-auslesen ---------------------------------
def read_and_map(xls: pd.ExcelFile, sheet_name: str, mapping: dict, usecols=None) -> pd.DataFrame:
    if sheet_name not in xls.sheet_names:
        
        # ----------------------------- leeres DF mit 0 Zeilen
        return pd.DataFrame()
    df = pd.read_excel(xls, sheet_name=sheet_name, usecols=usecols)
    df = _normalize_columns(df)

    # --------------------------------- Fehlerbehandlung Oxygen
    if "dissolved_oxygen_mg_per_l" in df.columns and "dissolved_oxigen_mg_per_l" not in df.columns:
        df = df.rename(columns={"dissolved_oxygen_mg_per_l": "dissolved_oxigen_mg_per_l"})

    ren = _available_mapping(df, mapping)
    if not ren:
        return pd.DataFrame()

    df = df.rename(columns=ren)
    return df[list(ren.values())]

In [10]:
def csv_to_excel(csv_path: Path, excel_path: Path, sheet_name: str = "SHEET1") -> None:

    # ------------------------------ CSV einlesen ---------------------------------
    df = pd.read_csv(csv_path, low_memory=False)

    # ------------------------- In Excel schreiben --------------------------------
    excel_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)

        # ------------------- Zugriff auf Arbeitsmappe -----------------------------
        workbook = writer.book
        worksheet = writer.sheets[sheet_name]

        
        # ---------------------- Kopfzeilen-Format --------------------------------
        header_format = workbook.add_format({
            "bold": True,
            "text_wrap": True,
            "valign": "center",
            "fg_color": "#D3D3D3",
            "border": 1
        })
        for col_num, value in enumerate(df.columns.values):
            worksheet.write(0, col_num, value, header_format)

        
        # ---------------------- Spaltenbreite dynamisch ---------------------------
        for i, col in enumerate(df.columns):
            max_len = max(df[col].astype(str).map(len).max(), len(col)) + 2
            worksheet.set_column(i, i, min(max_len, 50))  # begrenze max. Breite

    print(f"Excel-Pfad:\n{excel_path.resolve()}")

In [11]:
# ------------------------------- mg/L zu mmol/L umrechnung ------------------------------
def mgL_to_mmolL(series: pd.Series, M_gmol: float) -> pd.Series:
    # mg/L / (g/mol) = 1e-3 mol/L = mmol/L
    return _clean_numeric(series) / M_gmol

# ------------------------------- mg/L zu umol/L umrechnung ------------------------------
def mgL_to_umolL(series: pd.Series, M_gmol: float) -> pd.Series:
    # (mg/L / g/mol) * 1000 = µmol/L
    return mgL_to_mmolL(series, M_gmol) * 1000.0

In [12]:
# -------------------------- Entfernen von Leerzeichen und N/D Werten ---------------------
def _clean_numeric(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip()
         .str.replace(r"(?i)\bN/?D\b", "", regex=True)  
         .str.replace(r"[<>≈~]", "", regex=True)       
         .replace({"": np.nan})
         .pipe(pd.to_numeric, errors="coerce")
    )

<div class="alert alert-danger">
    <h1><b>DATENBANK 1</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.reflect-h2020.eu/efa/#form1 <br>
    <b>DATENBANK 1:</b> Komplette REFLECT Datenbank, stand 05.11.2025
</div>

In [13]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_reflect_path = Path.cwd() / "Zwischenstände" / "REFLECT_Datenbank"
print(f"REFLECT-Datenbank: {mid_reflect_path.resolve()}")

REFLECT-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank


In [14]:
# ----------------------------------- REFLECT-Pfad einlesen -----------------------------------
reflect_path = input_path / "REFLECT_Horizon_2020-Data_Export_2025-11-05_20-59-36.xlsx"


# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_reflect = pd.read_excel(reflect_path)
display(df_reflect.head())

,fluid_sample_id,well_id,local_id_f,latitude_,longitude_,country,powpl_name,nuts2_name,nuts3_name,date_of_well_completion_year,...,oxigen_18_so4_thousandths,sulfur_34_so4_thousandths,sulfur_34_h2s_thousandths,carbon_13_co2_thousandths,lithium_7_thousandths,boron_11_thousandths,strontium_87sr_per_86sr_thousandths,references_for_fluid_data,remarks_fluid,Database_number
0,W_AU_001_FS_001,W_AU_001,Blumau 2,47.12,16.04,Austria,N/D,Steiermark,Oststeiermark,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,Analysis GFZ laboratory,N/D,3
1,W_AU_001_FS_002,W_AU_001,Blumau 2,47.12,16.04,Austria,N/D,Steiermark,Oststeiermark,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,Analysis GFZ laboratory,N/D,3
2,W_AU_001_FS_003,W_AU_001,Blumau 2,47.12,16.04,Austria,N/D,Steiermark,Oststeiermark,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,Analysis GFZ laboratory,N/D,3
3,W_AU_001_FS_004,W_AU_001,Blumau 2,47.12,16.04,Austria,N/D,Steiermark,Oststeiermark,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,Hydroisotop GmbH 347619,N/D,3
4,W_AU_001_FS_005,W_AU_001,Blumau 2,47.12,16.04,Austria,N/D,Steiermark,Oststeiermark,N/D,...,N/D,N/D,N/D,N/D,N/D,N/D,N/D,Hydroisotop GmbH 347620,N/D,3


<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [15]:
# ----------------------------------- Konstanten -----------------------------------
SHEET_FLUID = "Fluid-well Data Export"
SHEET_ROCK  = "Rock-well Data Export"
SHEET_RES   = "Reservoir-well Data Export"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel ------------------------------
# Beachten: Spaltennamen immer KLEINSCHREIBEN, da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive
fluid_map = {
    "nuts2_name": "location",
    "local_id_f": "well_or_spring_name",
    "latitude_": "WGS84_latitude",
    "longitude_": "WGS84_longitude",
    "well_depth_m": "depth_bgl_in_m",
    "temperature_c": "temperature_in_c",
    "electrical_conductivity_us_per_cm_ec25": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "eh_mv": "redox_potential_in_mV",
    "tds_mg_per_l": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durhc Addition
    "dissolved_oxigen_mg_per_l": "O2_in_mmol/L",        
    "na_mg_per_l": "Na_in_mmol/L",
    "mg_mg_per_l": "Mg_in_mmol/L",
    "ca_mg_per_l": "Ca_in_mmol/L",
    "cl_mg_per_l": "Cl_in_mmol/L",
    "so4_mg_per_l": "SO4_in_mmol/L",
    "hco3_mg_per_l": "HCO3_in_mmol/L",
    "li_mg_per_l": "Li_in_mmol/L",
    "k_mg_per_l": "K_in_mmol/L",
    "sr_mg_per_l": "Sr_in_umol/L",
    "fe_mg_per_l": "Fe_in_mmol/L",
    "mn_mg_per_l": "Mn_in_mmol/L",
    "f_mg_per_l": "F_in_umol/L",
    "database_number": "Database_number",
}

rock_map = {
    "rock_type": "rock_type"
}

reservoir_map = {
    "age_of_formation": "stratigraphic_period"
}

In [16]:
# -------------------------------- Dateien/Ordner --------------------------------
mid_reflect_path = Path.cwd() / "Zwischenstände" / "REFLECT_Datenbank"
mid_reflect_path.mkdir(parents=True, exist_ok=True)


# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(reflect_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_fluid = read_and_map(xls, SHEET_FLUID, fluid_map)
df_rock  = read_and_map(xls, SHEET_ROCK, rock_map)
df_res   = read_and_map(xls, SHEET_RES, reservoir_map)


# -------------------------------- Zusammenführen --------------------------------
common_key = "well_or_spring_name"
can_merge = all([(common_key in df.columns) for df in [df_fluid, df_rock, df_res] if not df.empty])

if can_merge:
    df_combined = df_fluid.merge(df_rock, on=common_key, how="left").merge(df_res, on=common_key, how="left")
else:
    # --------------------------- Fallback -----------------------------------------
    df_combined = pd.concat([df_fluid, df_rock, df_res], axis=1)

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()


# ----------------------------- TDS behalten ---------------------------------------
if "total_dissolved_solids_in_mmol/L" not in df_combined.columns:
    df_combined["total_dissolved_solids_in_mmol/L"] = pd.NA

if "total_dissolved_solids_in_mg/L" not in df_combined.columns and "total_dissolved_solids_in_mg/L" in fluid_map.values():
    df_combined["total_dissolved_solids_in_mg/L"] = pd.NA


# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_reflect_path = mid_reflect_path / "before_conversion_reflect_mapped.csv"
df_out.to_csv(out_reflect_path, index=False, encoding="utf-8-sig")


print(f"REFLECT-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {out_reflect_path.resolve()}")

REFLECT-Mapping abgeschlossen: 5105 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank\before_conversion_reflect_mapped.csv


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [17]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_reflect_path / "before_conversion_reflect_mapped.csv"
excel_path = mid_reflect_path / "before_conversion_reflect_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank\before_conversion_reflect_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank\before_conversion_reflect_mapped_readable.xlsx


In [18]:
# Auskommentieren, wenn dies nur am Ende durchgeführt werden soll
# Einkommentieren, wenn Zwischenstände überprüft werden sollen

# csv_to_excel(csv_path, excel_path)

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [19]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_reflect_path, index=False, encoding="utf-8-sig")

In [20]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank\before_conversion_reflect_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [21]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}


# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan


In [22]:

# Umrechnungen (Annahme: Spalten liegen aktuell trotz Benennung mit mmol/L / umol/L in mg/L vor)
mmol_map = {
    "O2_in_mmol/L": "O2", "Na_in_mmol/L": "Na", "Mg_in_mmol/L": "Mg", "Ca_in_mmol/L": "Ca",
    "Cl_in_mmol/L": "Cl", "SO4_in_mmol/L": "SO4", "HCO3_in_mmol/L": "HCO3", "Li_in_mmol/L": "Li",
    "K_in_mmol/L": "K", "Fe_in_mmol/L": "Fe", "Mn_in_mmol/L": "Mn", "NO3_in_mmol/L": "NO3",
    "HS_in_mmol/L": "HS",
}
umol_map = {
    "Sr_in_umol/L": "Sr", "F_in_umol/L": "F", "NH4_in_umol/L": "NH4", "H2SiO3_in_umol/L": "H2SiO3",
}


# ------- mg/L -> mmol/L ---------
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L --------
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- TDS in mmol/L neu berechnen (O2 nicht inkludieren) -------------------------------------
tds_mmols_cols = [
    "Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Fe_in_mmol/L","Mn_in_mmol/L",
    "NO3_in_mmol/L","HS_in_mmol/L",
]
# Differenzierung für umol/L Spalten
tds_umols_cols = ["Sr_in_umol/L","F_in_umol/L","NH4_in_umol/L","H2SiO3_in_umol/L"]


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_reflect_path = mid_reflect_path / "after_conversion_reflect_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_reflect_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_reflect_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_reflect_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 5105 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank\after_conversion_reflect_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [23]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_reflect_path / "after_conversion_reflect_mapped.csv"
excel_path = mid_reflect_path / "after_conversion_reflect_mapped_readable.xlsx"

In [24]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\REFLECT_Datenbank\after_conversion_reflect_mapped_readable.xlsx


In [25]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_reflect_path / "after_conversion_reflect_mapped.csv"
excel_path = mid_reflect_path / "after_conversion_reflect_mapped_readable.xlsx"

In [26]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
reflect_output_path = output_path / "REFLECT_cleaned_Database.csv" 
reflect_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(reflect_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {reflect_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\REFLECT_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [27]:
map_path = base_dir / "Kartierung"
reflect_map_path = map_path / "REFLECT_Map.html" 

In [28]:
df = df_out.copy()

# --------------------------- Nur gültige Koordinaten -------------------------------
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# -------------------------- Kartenmittelpunkt setzen -------------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------------- Punkte clustern -----------------------------------
cluster = MarkerCluster().add_to(m)

for _, r in pts.iterrows():
    folium.CircleMarker(
        location=[r["WGS84_latitude"], r["WGS84_longitude"]],
        radius=5,
        fill=True,
        fill_opacity=0.9,
        weight=1,
        popup=folium.Popup(str(r.get("well_or_spring_name", "")), max_width=300),
        tooltip=str(r.get("well_or_spring_name", "")),
    ).add_to(cluster)

# --------------------------------- Optional ----------------------------------------
folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m  # Im Notebook wird die interaktive Karte direkt angezeigt
m.save(reflect_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 2</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.mdpi.com/2073-4441/14/1/132<br>
    <b>DATENBANK 2:</b>Hydrochemical Characteristics of Earthquake-Related Thermal Springs along the Weixi–Qiaohou Fault, Southeast Tibet Plateau , stand 06.11.2025<br>
</div>

In [29]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_mdpi_tibet_path = Path.cwd() / "Zwischenstände" / "MDPI_Tibet_Datenbank"
print(f"MDPI-Tibet-Datenbank: {mid_mdpi_tibet_path.resolve()}")

MDPI-Tibet-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank


In [30]:
# ----------------------------------- REFLECT-Pfad einlesen -----------------------------------
mdpi_tibet_path = input_path / "MDPI_Parsed.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_mdpi_tibet = pd.read_excel(mdpi_tibet_path)

# display(df_usgs.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [31]:
# ----------------------------------- Konstanten -----------------------------------
SHEET_1 = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Sheet1 = {
    "site": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "temperature_c": "temperature_in_c",
    "conductivity_us_cm": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "na_mgl": "Na_in_mmol/L",
    "mg_mgl": "Mg_in_mmol/L",
    "ca_mgl": "Ca_in_mmol/L",
    "cl_mgl": "Cl_in_mmol/L",
    "so4_mgl": "SO4_in_mmol/L",
    "hco3_mgl": "HCO3_in_mmol/L",
    "li_ugl": "Li_in_mmol/L",
    "k_mgl": "K_in_mmol/L",
    "sr_ugl": "Sr_in_umol/L",
    "fe_ugl": "Fe_in_mmol/L",
    "mn_ugl": "Mn_in_mmol/L",
    "f_mgl": "F_in_umol/L",
    "no3_mgl": "NO3_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [32]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_mdpi_tibet_path
# mdpi_tibet_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(mdpi_tibet_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, SHEET_1, Sheet1)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_mdpi_tibet_path = mid_mdpi_tibet_path / "before_conversion_mdpi_tibet_mapped.csv"
df_out.to_csv(out_mdpi_tibet_path, index=False, encoding="utf-8-sig")


print(f"USGS-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {out_mdpi_tibet_path.resolve()}")

USGS-Mapping abgeschlossen: 52 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank\before_conversion_mdpi_tibet_mapped.csv


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [33]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_mdpi_tibet_path / "before_conversion_mdpi_tibet_mapped.csv"
excel_path = mid_mdpi_tibet_path / "before_conversion_mdpi_tibet_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank\before_conversion_mdpi_tibet_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank\before_conversion_mdpi_tibet_mapped_readable.xlsx


In [34]:
# csv_to_excel(csv_path, excel_path)

<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung aller Werte in mg/L</b><br>
    Das manuelle Mapping hat ergeben, dass vier Parameter nicht in mg/L vorliegen:<br>
    - li_ugl (bereits umgeschrieben zu li_in_mmol/l)<br>
    - sr_ugl (bereits umgeschrieben zu sr_in_mmol/l)<br>
    - fe_ugl (bereits umgeschrieben zu fe_in_mmol/l)<br>
    - mn_ugl (bereits umgeschrieben zu mn_in_mmol/l)
</div>

In [35]:
# Diese Spalten enthalten eigentlich Werte in µg/L (falsch benannt)
ugl_cols = ["Li_in_mmol/L", "Sr_in_umol/L", "Fe_in_mmol/L", "Mn_in_mmol/L"]

for col in ugl_cols:
    if col in df_out.columns:
        # als Zahl interpretieren und von µg/L -> mg/L umrechnen
        df_out[col] = pd.to_numeric(df_out[col], errors="coerce") / 1000.0

In [36]:
# df_out.to_csv(out_mdpi_tibet_path, index=False, encoding="utf-8-sig")

In [37]:
# csv_to_excel(csv_path, excel_path)

<div class="alert alert-info">
    <b>Schritt 2.3 | Überprüfung</b><br>
    - Es wurde manuell überprüft, dass dieser Schritt tatsächlich nur diejenigen gewünschten Spalten umrechnet<br>
    - Anschließend liegen alle Werte korrekterweise in mg/L vor<br>
    - Die Funktion kann nun angewandt werden
</div>

<div class="alert alert-info">
    <b>Schritt 2.4 | Data Insertion</b><br>
    Es ist kein Name für die "location" gegeben - dieser soll jedoch ebenfalls angefügt werden<br>
    Wir verwenden das in der Publikation beschriebene Gebiet "Weixi–Qiaohou Fault"
</div>

In [38]:
df_out["location"] = "Weixi–Qiaohou Fault"
# display(df_out["location"] )
df_out.to_csv(out_mdpi_tibet_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [39]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_mdpi_tibet_path, index=False, encoding="utf-8-sig")

In [40]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank\before_conversion_mdpi_tibet_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [41]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [42]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_mdpi_tibet_path = mid_mdpi_tibet_path / "after_conversion_mdpi_tibet_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_mdpi_tibet_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_mdpi_tibet_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_mdpi_tibet_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 52 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank\after_conversion_mdpi_tibet_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [43]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_mdpi_tibet_path / "after_conversion_mdpi_tibet_mapped.csv"
excel_path = mid_mdpi_tibet_path / "after_conversion_mdpi_tibet_mapped_readable.xlsx"

In [44]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI_Tibet_Datenbank\after_conversion_mdpi_tibet_mapped_readable.xlsx


In [45]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
mdpi_tibet_output_path = output_path / "MDPI_Tibet_cleaned_Database.csv" 
mdpi_tibet_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(mdpi_tibet_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {mdpi_tibet_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\MDPI_Tibet_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [46]:
map_path = base_dir / "Kartierung"
mdpi_tibet_map_path = map_path / "MDPI_Tibet_Map.html" 

In [47]:
df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen ----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# --------------------- FastMarkerCluster mit Koordinaten füllen --------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(mdpi_tibet_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 3</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://gdr.openei.org/submissions/425<br>
    <b>DATENBANK 3:</b> CAES 2014 Chemical Analyses of Thermal Wells and Springs in Southeastern Idaho, stand 06.11.2025<br>
</div>

In [48]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_caes2014_path = Path.cwd() / "Zwischenstände" / "CAES2024_Datenbank"
print(f"CAES2014-Tibet-Datenbank: {mid_caes2014_path.resolve()}")

CAES2014-Tibet-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank


In [49]:
# ----------------------------------- REFLECT-Pfad einlesen -----------------------------------
caes2014_path = input_path / "CAES2014GeothermalData.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_caes2014 = pd.read_excel(caes2014_path)

# display(df_caes2014.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [50]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "Data"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "county": "location",
    "samplingfeaturename": "well_or_spring_name",
    "latdegree": "WGS84_latitude",
    "longdegree": "WGS84_longitude",
    "drilleddepth_ft": "depth_bgl_in_m",
    "fluidtemperature_c": "temperature_in_c",
    "ph_field": "pH",
    "na_mgl": "Na_in_mmol/L",
    "mg_mgl": "Mg_in_mmol/L",
    "ca_mgl": "Ca_in_mmol/L",
    "cl_mgl": "Cl_in_mmol/L",
    "so4_mgl": "SO4_in_mmol/L",
    "alkalinity_mgl": "HCO3_in_mmol/L",
    "li_mgl": "Li_in_mmol/L",
    "k_mgl": "K_in_mmol/L",
    "sr_mgl": "Sr_in_umol/L",
    "f_mgl": "F_in_umol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [51]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_caes2014_path
# caes2014_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(caes2014_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_caes2014_path = mid_caes2014_path / "before_conversion_caes2014_mapped.csv"
df_out.to_csv(out_caes2014_path, index=False, encoding="utf-8-sig")


print(f"USGS-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_caes2014_path.resolve()}")

USGS-Mapping abgeschlossen: 42 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [52]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_caes2014_path / "before_conversion_caes2014_mapped.csv"
excel_path = mid_caes2014_path / "before_conversion_caes2014_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank\before_conversion_caes2014_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank\before_conversion_caes2014_mapped_readable.xlsx


In [53]:
# csv_to_excel(csv_path, excel_path)

<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung von DrilledDepth_ft in Meter/L</b><br>
    - drilleddepth_ft muss in m umgewandelt werden (ist aktuell in depth_bgl_in_m gespeichert)
</div>

In [54]:
df_out["depth_bgl_in_m"] = pd.to_numeric(df_out["depth_bgl_in_m"], errors="coerce") * 0.3048

In [55]:
df_out.to_csv(out_caes2014_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [56]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_caes2014_path, index=False, encoding="utf-8-sig")

In [57]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank\before_conversion_caes2014_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [58]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [59]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_caes2014_path = mid_caes2014_path / "after_conversion_caes2014_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_caes2014_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_caes2014_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_caes2014_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 42 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank\after_conversion_caes2014_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [60]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_caes2014_path / "after_conversion_caes2014_mapped.csv"
excel_path = mid_caes2014_path / "after_conversion_caes2014_mapped_readable.xlsx"

In [61]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\CAES2024_Datenbank\after_conversion_caes2014_mapped_readable.xlsx


In [62]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
caes2014_output_path = output_path / "CAES2014_cleaned_Database.csv" 
caes2014_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(caes2014_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {caes2014_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\CAES2014_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [63]:
map_path = base_dir / "Kartierung"
caes2014_map_path = map_path / "CAES2014_Map.html" 

In [64]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(caes2014_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 4</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://doi.pangaea.de/10.1594/PANGAEA.916088#:~:text=DGS,UNMIG%29.%20Italy.%20%28ht<br>
    <b>DATENBANK 4:</b> Alpen, stand 6.11.2025<br>
    -> Geospatial data and estimated circulation temperature, heat-flux and thermal footprint for thermal springs in the Alps [dataset]
</div>

<div class="alert alert-info">
    <b>Schritt 0 | Erstellen einer Excel</b><br>
    Da die Datei direkt in einer .csv vorliegt, wird diese zuerst in eine Excel-Datei umgewandelt, um übersichtlicher feststellen zu können, wie das Mapping ablaufen muss
</div>

In [65]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_alps_path = Path.cwd() / "Zwischenstände" / "ALPS_Datenbank"
print(f"ALPS-Datenbank: {mid_alps_path.resolve()}")

ALPS-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank


In [66]:
# ----------------------------------- Pfad einlesen -----------------------------------
alps_path = input_path / "thermal_springs_alps_with_HF_estimates.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_alps = pd.read_excel(alps_path)

# display(df_alps.head())

In [67]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_alps_path / "before_first_transformation_alps.csv"
excel_path = mid_alps_path / "after_first_transformation_alps.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\before_first_transformation_alps.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\after_first_transformation_alps.xlsx


In [68]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\after_first_transformation_alps.xlsx


<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [69]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "SHEET1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "spring_location": "location",
    "spring_name": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "well_depth_m": "depth_bgl_in_m",
    "lithology": "rock_type",
    "temperature_degr_c": "temperature_in_c",
    "ec_s_m-1": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "tds_mg_l-1": "total_dissolved_solids_in_mmol/l",
    "na_mg_l-1": "Na_in_mmol/L",
    "mg_mg_l-1": "Mg_in_mmol/L",
    "ca_mg_l-1": "Ca_in_mmol/L",
    "cl_mg_l-1": "Cl_in_mmol/L",
    "so4_mg_l-1": "SO4_in_mmol/L",
    "hco3_mg_l-1": "HCO3_in_mmol/L",
    "li_mg_l-1": "Li_in_mmol/L",
    "k_mg_l-1": "K_in_mmol/L",
    "nh4_mg_l-1": "NH4_in_umol/L",
    "f_mg_l-1": "F_in_umol/L",
    "no3_mg_l-1": "NO3_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [70]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_alps_path
# alps_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(alps_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_alps_path = mid_alps_path / "before_conversion_alps_mapped.csv"
df_out.to_csv(out_alps_path, index=False, encoding="utf-8-sig")


print(f"ALPS-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_alps_path.resolve()}")

ALPS-Mapping abgeschlossen: 311 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [71]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_alps_path / "before_conversion_alps_mapped.csv"
excel_path = mid_alps_path / "before_conversion_alps_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\before_conversion_alps_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\before_conversion_alps_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung von "EC_(S_m^-1)" in uS/m</b><br>
    - drilleddepth_ft muss von S/m in uS/m umgewandelt werden (ist aktuell in "electrical_conductivity_25c_in_uS/cm" gespeichert)
</div>

In [72]:
df_out["electrical_conductivity_25c_in_uS/cm"] = pd.to_numeric(df_out["electrical_conductivity_25c_in_uS/cm"], errors="coerce") * 100

In [73]:
df_out.to_csv(out_alps_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [74]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_alps_path, index=False, encoding="utf-8-sig")

In [75]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\before_conversion_alps_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [76]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [77]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_alps_path = mid_alps_path / "after_conversion_alps_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_alps_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_alps_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_alps_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 311 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\after_conversion_alps_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [78]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_alps_path / "after_conversion_alps_mapped.csv"
excel_path = mid_alps_path / "after_conversion_alps_mapped_readable.xlsx"

In [79]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ALPS_Datenbank\after_conversion_alps_mapped_readable.xlsx


In [80]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
alps_output_path = output_path / "ALPS_cleaned_Database.csv" 
alps_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(alps_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {alps_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\ALPS_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [81]:
map_path = base_dir / "Kartierung"
alps_map_path = map_path / "ALPS_Map.html" 

In [82]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(alps_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 5</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://open.yukon.ca/data/geothermal-thermal-springs<br>
    <b>DATENBANK 5:</b> YUKON, stand 12.11.2025<br>
    -> Geothermal Thermal Springs 
</div>

<div class="alert alert-info">
    <b>Schritt 0 | Erstellen einer Excel</b><br>
    Da die Datei direkt in einer .csv vorliegt, wird diese zuerst in eine Excel-Datei umgewandelt, um übersichtlicher feststellen zu können, wie das Mapping ablaufen muss
</div>

In [83]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_yukon_path = Path.cwd() / "Zwischenstände" / "YUKON_Datenbank"
print(f"YUKON-Datenbank: {mid_yukon_path.resolve()}")

YUKON-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank


In [84]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
yukon_path = input_path / "Yukon_Geothermal_Springs.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_yukon = pd.read_excel(yukon_path)

# display(df_yukon.head())

In [85]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_yukon_path / "before_first_transformation_yukon.csv"
excel_path = mid_yukon_path / "after_first_transformation_yukon.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\before_first_transformation_yukon.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\after_first_transformation_yukon.xlsx


In [86]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\after_first_transformation_yukon.xlsx


<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [87]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "SHEET1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "location": "location",
    "name": "well_or_spring_name",
    "latitude_dd": "WGS84_latitude",
    "longitude_dd": "WGS84_longitude",
    "conductivity": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "na_mg_l": "Na_in_mmol/L",
    "mg_mg_l": "Mg_in_mmol/L",
    "ca_mg_l": "Ca_in_mmol/L",
    "cl_mg_l": "Cl_in_mmol/L",
    "so4_mg_l": "SO4_in_mmol/L",
    "hco3_mg_l": "HCO3_in_mmol/L",
    "k_mg_l": "K_in_mmol/L",
    "f_mg_l": "F_in_umol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [88]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_yukon_path
# yukon_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(yukon_path)

# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_yukon_path = mid_yukon_path / "before_conversion_yukon_mapped.csv"
df_out.to_csv(out_yukon_path, index=False, encoding="utf-8-sig")


print(f"YUKON-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_yukon_path.resolve()}")

YUKON-Mapping abgeschlossen: 88 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [89]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_yukon_path / "before_conversion_yukon_mapped.csv"
excel_path = mid_yukon_path / "before_conversion_yukon_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\before_conversion_yukon_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\before_conversion_yukon_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [90]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_yukon_path, index=False, encoding="utf-8-sig")

In [91]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\before_conversion_yukon_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [92]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [93]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_yukon_path = mid_yukon_path / "after_conversion_yukon_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_yukon_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_yukon_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_yukon_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 88 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\after_conversion_yukon_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [94]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_yukon_path / "after_conversion_yukon_mapped.csv"
excel_path = mid_yukon_path / "after_conversion_yukon_mapped_readable.xlsx"

In [95]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\YUKON_Datenbank\after_conversion_yukon_mapped_readable.xlsx


In [96]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
yukon_output_path = output_path / "YUKON_cleaned_Database.csv" 
yukon_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(yukon_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {yukon_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\YUKON_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [97]:
map_path = base_dir / "Kartierung"
yukon_map_path = map_path / "YUKON_Map.html" 

In [98]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(yukon_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 6</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://repositorio.uchile.cl/bitstream/handle/2250/149568/Geochemistry-of-thermal-waters.pdf<br>
    <b>DATENBANK 6:</b> CHILE Datenbank, stand 13.11.2025, Geochemistry of thermal waters in the Southern Volcanic Zone, Chile –
Implications for structural controls on geothermal fluid composition
</div>

In [99]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_chile_path = Path.cwd() / "Zwischenstände" / "SVZ-CHILE_Datenbank"
print(f"CHILE-Datenbank: {mid_chile_path.resolve()}")

CHILE-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank


In [100]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
chile_path = input_path / "SVZ_Chile_Table1_merged.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_chile = pd.read_excel(chile_path)

# display(df_chile.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [101]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "TABLE1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "location_name": "location",
    "fault_domain": "well_or_spring_name",
    "wgs84_latitude": "WGS84_latitude",
    "wgs84_longitude": "WGS84_longitude",
    "t_c": "temperature_in_c",
    "ph": "pH",
    "tds_ppm": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durhc Addition 
    "na_ppm": "Na_in_mmol/L",
    "mg_ppm": "Mg_in_mmol/L",
    "ca_ppm": "Ca_in_mmol/L",
    "cl_ppm": "Cl_in_mmol/L",
    "so4_ppm": "SO4_in_mmol/L",
    "hco3_ppm": "HCO3_in_mmol/L",
    "li_ppm": "Li_in_mmol/L",
    "k_ppm": "K_in_mmol/L",
    "sr_ppb": "Sr_in_umol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [102]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_chile_path
# chile_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(chile_path)



# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# display(df_sheet1.head())
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()

display(df_out.head())
# ------------------------------ Export ---------------------------------------------
out_chile_path = mid_chile_path / "before_conversion_chile_mapped.csv"
df_out.to_csv(out_chile_path, index=False, encoding="utf-8-sig")


print(f"CHILE-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_chile_path.resolve()}")

,location,well_or_spring_name,WGS84_latitude,WGS84_longitude,depth_bgl_in_m,rock_type,stratigraphic_period,temperature_in_c,electrical_conductivity_25c_in_uS/cm,pH,...,K_in_mmol/L,Sr_in_umol/L,NH4_in_umol/L,Fe_in_mmol/L,Mn_in_mmol/L,F_in_umol/L,NO3_in_mmol/L,H2SiO3_in_umol/L,HS_in_mmol/L,Database_number
0,Nevado de Chillan Vn.,ATF,-36.86333,-71.37667,<NA>,<NA>,<NA>,68.0,<NA>,3.9,...,21.48,78.6,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,13
1,Nevado de Chillan Vn.,ATF,-36.86333,-71.37667,<NA>,<NA>,<NA>,82.0,<NA>,2.6,...,4.45,47.5,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,13
2,Nevado de Chillan Vn.,ATF,-36.86333,-71.37667,<NA>,<NA>,<NA>,91.0,<NA>,2.4,...,81.30,143.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,13
3,Trapa Trapa,LOFS,-37.82667,-71.13000,<NA>,<NA>,<NA>,45.0,<NA>,7.8,...,1.70,305.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,13
4,Copahue Vn.,LOFS,-37.85000,-71.17000,<NA>,<NA>,<NA>,87.0,<NA>,2.7,...,35.50,141.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,13


CHILE-Mapping abgeschlossen: 30 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [103]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_chile_path / "before_conversion_chile_mapped.csv"
excel_path = mid_chile_path / "before_conversion_chile_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank\before_conversion_chile_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank\before_conversion_chile_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung von "Li", welches aktuell in umol/L vorliegt zu mmol/L</b><br>
    - ist aktuell in "Li_in_mmol/L" gespeichert
</div>

In [104]:
df_out["Li_in_mmol/L"] = pd.to_numeric(df_out["Li_in_mmol/L"], errors="coerce") / 1000

In [105]:
df_out.to_csv(out_chile_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [106]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_chile_path, index=False, encoding="utf-8-sig")

In [107]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank\before_conversion_chile_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [108]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [109]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_chile_path = mid_chile_path / "after_conversion_chile_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_chile_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_chile_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_chile_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 30 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank\after_conversion_chile_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [110]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_chile_path / "after_conversion_chile_mapped.csv"
excel_path = mid_chile_path / "after_conversion_chile_mapped_readable.xlsx"

In [111]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\SVZ-CHILE_Datenbank\after_conversion_chile_mapped_readable.xlsx


In [112]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
chile_output_path = output_path / "SVZ-CHILE_cleaned_Database.csv" 
chile_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(chile_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {chile_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\SVZ-CHILE_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [113]:
map_path = base_dir / "Kartierung"
chile_map_path = map_path / "SVZ-Chile_Map.html" 

In [114]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(chile_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 7</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.cell.com/heliyon/fulltext/S2405-8440(23)04561-9?_returnURL=https%3A%2F%2Flinkinghub.elsevier.com%2Fretrieve%2Fpii%2FS2405844023045619%3Fshowall%3Dtrue<br>
    <b>DATENBANK 7:</b> GANDAKI Datenbank, stand 10.11.2025, Gandaki Province, Nepal 
</div>

In [115]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_gandaki_path = Path.cwd() / "Zwischenstände" / "GANDAKI_Datenbank"
print(f"GANDAKI-Datenbank: {mid_gandaki_path.resolve()}")

GANDAKI-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank


In [116]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
gandaki_path = input_path / "mmc1_updated.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_gandaki = pd.read_excel(gandaki_path)

# display(df_gandaki.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [117]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "location": "location",
    "id": "well_or_spring_name",
    "wgs84_latitude": "WGS84_latitude",
    "wgs84_longitude": "WGS84_longitude",
    "temp_c": "temperature_in_c",
    "ec_us_cm": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "tds_mg_l": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durhc Addition 
    "na_mg_l": "Na_in_mmol/L",
    "mg_mg_l": "Mg_in_mmol/L",
    "ca_mg_l": "Ca_in_mmol/L",
    "cl_mg_l": "Cl_in_mmol/L",
    "so4_mg_l": "SO4_in_mmol/L",
    "hco3_mg_l": "HCO3_in_mmol/L",
    "k_mg_l": "K_in_mmol/L",
    "fe_mg_l": "Fe_in_mmol/L",
    "f_mg_l": "F_in_umol/L",
    "no3_mg_l": "NO3_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [118]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_gandaki_path
# gandaki_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(gandaki_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_gandaki_path = mid_gandaki_path / "before_conversion_gandaki_mapped.csv"
df_out.to_csv(out_gandaki_path, index=False, encoding="utf-8-sig")


print(f"GANDAKI-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_gandaki_path.resolve()}")

GANDAKI-Mapping abgeschlossen: 12 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [119]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_gandaki_path / "before_conversion_gandaki_mapped.csv"
excel_path = mid_gandaki_path / "before_conversion_gandaki_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank\before_conversion_gandaki_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank\before_conversion_gandaki_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [120]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_gandaki_path, index=False, encoding="utf-8-sig")

In [121]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank\before_conversion_gandaki_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [122]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [123]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_gandaki_path = mid_gandaki_path / "after_conversion_gandaki_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_gandaki_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_gandaki_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_gandaki_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 12 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank\after_conversion_gandaki_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [124]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_gandaki_path / "after_conversion_gandaki_mapped.csv"
excel_path = mid_gandaki_path / "after_conversion_gandaki_mapped_readable.xlsx"

In [125]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\GANDAKI_Datenbank\after_conversion_gandaki_mapped_readable.xlsx


In [126]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
gandaki_output_path = output_path / "GANDAKI_cleaned_Database.csv" 
gandaki_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(gandaki_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {gandaki_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\GANDAKI_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [127]:
map_path = base_dir / "Kartierung"
gandaki_map_path = map_path / "GANDAKI_Map.html" 

In [128]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(gandaki_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 8</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://essd.copernicus.org/articles/13/4847/2021/<br>
    <b>DATENBANK 8:</b> HESSEN Datenbank, stand 10.11.2025, HESSEN_Datenbank
</div>

In [129]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_hessen_path = Path.cwd() / "Zwischenstände" / "HESSEN_Datenbank"
print(f"HESSEN-Datenbank: {mid_hessen_path.resolve()}")

HESSEN-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank


In [130]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
hessen_path = input_path / "Hydrochemical_Database_Hessen_3D_2-0_RSc_202109.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_hessen = pd.read_excel(hessen_path)

# display(df_hessen.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [131]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "town": "location",
    "well_or_spring_name": "well_or_spring_name",
    "coordinates_r": "WGS84_latitude",
    "coordinates_h": "WGS84_longitude",
    "final_depth": "depth_bgl_in_m",
    "rock_type": "rock_type",
    "period": "stratigraphic_period",
    "temperature": "temperature_in_c",
    "electric": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "redox": "redox_potential_in_mV",
    "total_dissolved_solids_calculated": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durhc Addition 
    "oxigen": "O2_in_mmol/L",
    "sodium": "Na_in_mmol/L",
    "magnesium": "Mg_in_mmol/L",
    "calcium": "Ca_in_mmol/L",
    "chloride": "Cl_in_mmol/L",
    "sulphate": "SO4_in_mmol/L",
    "bicarbonate": "HCO3_in_mmol/L",
    "lithium": "Li_in_mmol/L",
    "potassium": "K_in_mmol/L",
    "strontium": "Sr_in_umol/L",
    "ammonium": "NH4_in_umol/L",
    "iron": "Fe_in_mmol/L",
    "manganese": "Mn_in_mmol/L",
    "fluoride": "F_in_umol/L",
    "nitrate": "NO3_in_mmol/L",
    "silica": "H2SiO3_in_umol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [132]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_hessen_path
# hessen_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(hessen_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_hessen_path = mid_hessen_path / "before_conversion_hessen_mapped.csv"
df_out.to_csv(out_hessen_path, index=False, encoding="utf-8-sig")


print(f"AAAAAAAAA-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_hessen_path.resolve()}")

AAAAAAAAA-Mapping abgeschlossen: 1035 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [133]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_hessen_path / "before_conversion_hessen_mapped.csv"
excel_path = mid_hessen_path / "before_conversion_hessen_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank\before_conversion_hessen_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank\before_conversion_hessen_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [134]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_hessen_path, index=False, encoding="utf-8-sig")

In [135]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank\before_conversion_hessen_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [136]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [137]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_hessen_path = mid_hessen_path / "after_conversion_hessen_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_hessen_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_hessen_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_hessen_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 1035 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank\after_conversion_hessen_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [138]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_hessen_path / "after_conversion_hessen_mapped.csv"
excel_path = mid_hessen_path / "after_conversion_hessen_mapped_readable.xlsx"

In [139]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\HESSEN_Datenbank\after_conversion_hessen_mapped_readable.xlsx


In [140]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
hessen_output_path = output_path / "HESSEN_cleaned_Database.csv" 
hessen_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(hessen_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {hessen_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\HESSEN_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [141]:
map_path = base_dir / "Kartierung"
hessen_map_path = map_path / "HESSEN_Map.html" 

In [142]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(hessen_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 9</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.mdpi.com/2073-4441/14/4/550#:~:text=The%2043%20hot%20springs%20were,rate%20of%20slip%20in%20the<br>
    <b>DATENBANK 9:</b> MDPI Xianshuihe Datenbank, stand 12.11.2025
</div>

In [143]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_xian_path = Path.cwd() / "Zwischenstände" / "MDPI-Xianshuihe_Datenbank"
print(f"XIANSHUIHE-Datenbank: {mid_xian_path.resolve()}")

XIANSHUIHE-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank


In [144]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
xian_path = input_path / "MPDI_Xianshuihe.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_xian = pd.read_excel(xian_path)

# display(df_xian.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [145]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "no": "well_or_spring_name",
    "wgs84_latitude": "WGS84_latitude",
    "wgs84_longitude": "WGS84_longitude",
    "temperature": "temperature_in_c",
    "ec": "electrical_conductivity_25c_in_uS/cm",
    "ph": "pH",
    "tds": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durhc Addition 
    "na": "Na_in_mmol/L",
    "mg": "Mg_in_mmol/L",
    "ca": "Ca_in_mmol/L",
    "cl": "Cl_in_mmol/L",
    "so4": "SO4_in_mmol/L",
    "hco3": "HCO3_in_mmol/L",
    "k": "K_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [146]:
# -------------------------------- Dateien/Ordner --------------------------------
# mix_xian_path
# xian_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(xian_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_xian_path = mid_xian_path / "before_conversion_xian_mapped.csv"
df_out.to_csv(out_xian_path, index=False, encoding="utf-8-sig")


print(f"XIAN-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_xian_path.resolve()}")

XIAN-Mapping abgeschlossen: 280 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [147]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_xian_path / "before_conversion_xian_mapped.csv"
excel_path = mid_xian_path / "before_conversion_xian_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank\before_conversion_xian_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank\before_conversion_xian_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [148]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_xian_path, index=False, encoding="utf-8-sig")

In [149]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank\before_conversion_xian_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [150]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [151]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_xian_path = mid_xian_path / "after_conversion_xian_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_xian_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_xian_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_xian_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 280 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank\after_conversion_xian_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [152]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_xian_path / "after_conversion_xian_mapped.csv"
excel_path = mid_xian_path / "after_conversion_xian_mapped_readable.xlsx"

In [153]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\MDPI-Xianshuihe_Datenbank\after_conversion_xian_mapped_readable.xlsx


In [154]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
xian_output_path = output_path / "XIANSHUIHE_cleaned_Database.csv" 
xian_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(xian_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {xian_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\XIANSHUIHE_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [155]:
map_path = base_dir / "Kartierung"
xian_map_path = map_path / "XIANSHUIHE_Map.html" 

In [156]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(xian_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 10</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.nature.com/articles/s41598-021-03701-1?error=cookies_not_supported&code=d245a11e-6034-45f3-b6f9-c53fc736651e#article-info<br>
    <b>DATENBANK 10:</b> Australia Mataranka, stand 12.11.2025
</div>

In [157]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_australia_path = Path.cwd() / "Zwischenstände" / "AUSTRALIA-MATARANKA_Datenbank"
print(f"AUSTRALIA-Datenbank: {mid_australia_path.resolve()}")

AUSTRALIA-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank


In [158]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
australia_path = input_path / "Australia_Mataranka.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_australia = pd.read_excel(australia_path)

# display(df_australia.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [159]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "cla_flow_path": "location",
    "bore_spring": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "na_mg-l": "Na_in_mmol/L",
    "mg_mg-l": "Mg_in_mmol/L",
    "ca_mg-l": "Ca_in_mmol/L",
    "cl_mg-l": "Cl_in_mmol/L",
    "so4_mg-l": "SO4_in_mmol/L",
    "alkalinity_meq-l": "HCO3_in_mmol/L",
    "k_mg-l": "K_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [160]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_australia_path
# australia_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(australia_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_australia_path = mid_australia_path / "before_conversion_australia_mapped.csv"
df_out.to_csv(out_australia_path, index=False, encoding="utf-8-sig")


print(f"AAAAAAAAA-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_australia_path.resolve()}")

AAAAAAAAA-Mapping abgeschlossen: 84 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [161]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_australia_path / "before_conversion_australia_mapped.csv"
excel_path = mid_australia_path / "before_conversion_australia_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank\before_conversion_australia_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank\before_conversion_australia_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung von "alkalinity_meq-l" in mg/L</b><br>
    - ist aktuell in "HCO3_in_mmol/L" gespeichert | Faktor 61.0168
</div>

In [162]:
df_out["HCO3_in_mmol/L"] = pd.to_numeric(df_out["HCO3_in_mmol/L"], errors="coerce") * 61.0168

In [163]:
df_out.to_csv(out_australia_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [164]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_australia_path, index=False, encoding="utf-8-sig")

In [165]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank\before_conversion_australia_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [166]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [167]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_australia_path = mid_australia_path / "after_conversion_australia_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_australia_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_australia_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_australia_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 84 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank\after_conversion_australia_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [168]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_australia_path / "after_conversion_australia_mapped.csv"
excel_path = mid_australia_path / "after_conversion_australia_mapped_readable.xlsx"

In [169]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\AUSTRALIA-MATARANKA_Datenbank\after_conversion_australia_mapped_readable.xlsx


In [170]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
australia_output_path = output_path / "AUSTRALIA-MATARANKA_cleaned_Database.csv" 
australia_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(australia_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {australia_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\AUSTRALIA-MATARANKA_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [171]:
map_path = base_dir / "Kartierung"
australia_map_path = map_path / "AUSTRALIA-MATARANKA_Map.html" 

In [172]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(australia_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK 11</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.tshe.org/ea/pdf/EA14(3)/EA14(3)_04.pdf#:~:text=Geochemical%20data%20from%20more%20than,The%20geochemical%20data<br>
    <b>DATENBANK 11:</b> THAILAND Datenbank, stand 12.11.2025
</div>

In [173]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_thailand_path = Path.cwd() / "Zwischenstände" / "THAILAND_Datenbank"
print(f"THAILAND-Datenbank: {mid_thailand_path.resolve()}")

THAILAND-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank


In [174]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
thailand_path = input_path / "Southern_Thailand_Parsed.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_thailand = pd.read_excel(thailand_path)

# display(df_thailand.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [175]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "geothermal_province": "location",
    "hot_spring": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "ph": "pH",
    "na_mg-l": "Na_in_mmol/L",
    "mg_mg-l": "Mg_in_mmol/L",
    "ca_mg-l": "Ca_in_mmol/L",
    "cl_mg-l": "Cl_in_mmol/L",
    "so4_mg-l": "SO4_in_mmol/L",
    "hco3_mg-l": "HCO3_in_mmol/L",
    "k_mg-l": "K_in_mmol/L",
    "fe_mg-l": "Fe_in_mmol/L",
    "mn_mg-l": "Mn_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [176]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_thailand_path
# thailand_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(thailand_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_thailand_path = mid_thailand_path / "before_conversion_thailand_mapped.csv"
df_out.to_csv(out_thailand_path, index=False, encoding="utf-8-sig")


print(f"AAAAAAAAA-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_thailand_path.resolve()}")

AAAAAAAAA-Mapping abgeschlossen: 31 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [177]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_thailand_path / "before_conversion_thailand_mapped.csv"
excel_path = mid_thailand_path / "before_conversion_thailand_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank\before_conversion_thailand_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank\before_conversion_thailand_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [178]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_thailand_path, index=False, encoding="utf-8-sig")

C:\Users\lucca\AppData\Local\Temp\ipykernel_23256\1240268547.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)


In [179]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank\before_conversion_thailand_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [180]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [181]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_thailand_path = mid_thailand_path / "after_conversion_thailand_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_thailand_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_thailand_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_thailand_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 31 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank\after_conversion_thailand_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [182]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_thailand_path / "after_conversion_thailand_mapped.csv"
excel_path = mid_thailand_path / "after_conversion_thailand_mapped_readable.xlsx"

In [183]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\THAILAND_Datenbank\after_conversion_thailand_mapped_readable.xlsx


In [184]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
thailand_output_path = output_path / "THAILAND_cleaned_Database.csv" 
thailand_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(thailand_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {thailand_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\THAILAND_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [185]:
map_path = base_dir / "Kartierung"
thailand_map_path = map_path / "THAILAND_Map.html" 

In [186]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(thailand_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK VORLETZTE</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://catalog.data.gov/dataset/argonne-geothermal-geochemical-database-v2-0-53978/resource/35bedd29-61f0-418e-8233-0511640a8d9e#:~:text=Argonne%20Geothermal%20Geochemical%20Database%20v2_00<br>
    <b>DATENBANK VORLETZTE:</b> ARGONNE Datenbank, stand 12.11.2025, Argonne Geothermal Geochemical Database v2_00.xlsx
</div>

In [187]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_argonne_path = Path.cwd() / "Zwischenstände" / "ARGONNE_Datenbank"
print(f"ARGONNE-Datenbank: {mid_argonne_path.resolve()}")

ARGONNE-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank


In [188]:
# ------------------------------------------ Pfad einlesen ------------------------------------------
argonne_path = input_path / "Argonne_Geothermal_Geochemical_Database_v2_00.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_argonne = pd.read_excel(argonne_path)

# display(df_argonne.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [189]:
# ----------------------------------- Konstanten -----------------------------------
DATA = "ALLDATA"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, keine Sonderzeichen außer -/_, keine Leerzeichen
# da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive

Data = {
    "state": "location",
    "name": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "depth_ft": "depth_bgl_in_m",
    "temperature_c": "temperature_in_c",
    "ph": "pH",
    "tds": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durhc Addition 
    "na": "Na_in_mmol/L",
    "mg": "Mg_in_mmol/L",
    "ca": "Ca_in_mmol/L",
    "cl": "Cl_in_mmol/L",
    "so4": "SO4_in_mmol/L",
    "hco3": "HCO3_in_mmol/L",
    "li": "Li_in_mmol/L",
    "k": "K_in_mmol/L",
    "sr": "Sr_in_umol/L",
    "nh4_n": "NH4_in_umol/L",
    "fe": "Fe_in_mmol/L",
    "mn": "Mn_in_mmol/L",
    "f": "F_in_umol/L",
    "no3_n": "NO3_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [190]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_argonne_path
# argonne_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(argonne_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, DATA, Data)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_argonne_path = mid_argonne_path / "before_conversion_argonne_mapped.csv"
df_out.to_csv(out_argonne_path, index=False, encoding="utf-8-sig")


print(f"ARGONNE-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {mid_argonne_path.resolve()}")

ARGONNE-Mapping abgeschlossen: 51078 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [191]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_argonne_path / "before_conversion_argonne_mapped.csv"
excel_path = mid_argonne_path / "before_conversion_argonne_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank\before_conversion_argonne_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank\before_conversion_argonne_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 2.3 | Umwandlung von "depth_ft" in m</b><br>
    - drilleddepth_ft muss von ft in m umgewandelt werden (ist aktuell in "depth_bgl_in_m" gespeichert)
</div>

In [192]:
df_out["depth_bgl_in_m"] = pd.to_numeric(df_out["depth_bgl_in_m"], errors="coerce") * 0.3048

In [193]:
df_out.to_csv(out_argonne_path, index=False, encoding="utf-8-sig")

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [194]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_argonne_path, index=False, encoding="utf-8-sig")

In [195]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank\before_conversion_argonne_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [196]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path, low_memory=False)


# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan

In [197]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_argonne_path = mid_argonne_path / "after_conversion_argonne_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_argonne_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_argonne_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_argonne_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 51078 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank\after_conversion_argonne_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [198]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_argonne_path / "after_conversion_argonne_mapped.csv"
excel_path = mid_argonne_path / "after_conversion_argonne_mapped_readable.xlsx"

In [199]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\ARGONNE_Datenbank\after_conversion_argonne_mapped_readable.xlsx


In [200]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
argonne_output_path = output_path / "ARGONNE_cleaned_Database.csv" 
argonne_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(argonne_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {argonne_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\ARGONNE_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [201]:
map_path = base_dir / "Kartierung"
argonne_map_path = map_path / "Argonne_Map.html" 

In [202]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ----------------------------- Kartenmittelpunkt setzen -----------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ------------------------ FastMarkerCluster mit Koordinaten ------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(argonne_map_path)

<div class="alert alert-danger">
    <h1><b>DATENBANK FINAL</b></h1>
</div>

<div class="alert alert-block alert-warning">
    https://www.sciencebase.gov/catalog/item/59d25d63e4b05fe04cc235f9<br>
    <b>DATENBANK final:</b> Komplette USGS Datenbank, stand 07.11.2025<br>
    -> Kommt an letzter Stelle, da lange Ladezeiten (auf Grund von über 100.000 Datenzeilen)
</div>

In [203]:
# --------------- Ordner für Zwischenstände definieren ------------------
mid_usgs_path = Path.cwd() / "Zwischenstände" / "USGS_Datenbank"
print(f"USGS-Datenbank: {mid_usgs_path.resolve()}")

USGS-Datenbank: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank


In [204]:
# ----------------------------------- REFLECT-Pfad einlesen -----------------------------------
usgs_path = input_path / "USGSPWDBv2.3n.xlsx"

# -------------------------------------- Excel-Datei einlesen ---------------------------------------
df_usgs = pd.read_excel(usgs_path)

# display(df_usgs.head())

<div class="alert alert-info">
    <b>Schritt 1 | Manuelles finden der zu verwendenden Daten</b><br>
    Findet von Hand statt - es wird manuell überprüft, welche Spalten des globalen Zielschemas, den Spalten in der Datenbank entsprechen.<br>
    Das Ergebnis dieser Auswertung findet sich in dem Ordner "Zwischenstände" der jeweiligen Datenbank in Form einer Mapping .pdf wieder
</div>

<div class="alert alert-info">
    <b>Schritt 2.1 | Mapping zu Datenschema als .csv Datei</b><br>
    Nachfolgend findet das Mapping der zu verwenden Daten im Bezug auf das globale Datenschema statt. <br>
    Dabei wird die angefertigte .pdf verwendet um die zu mappenden Daten übertragen zu können.<br><br>
    Falls eine Datenspalte nicht vorhanden ist, wird diese ergänzt, jedoch nicht ausgefüllt.
</div>

In [205]:
# ----------------------------------- Konstanten -----------------------------------
SHEET_1 = "Sheet1"

# --------------------------- Zielspalten wie gehabt -------------------------------
TARGET_COLUMNS = [
    "location",
    "well_or_spring_name",
    "WGS84_latitude",
    "WGS84_longitude",
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period",
    "temperature_in_c",
    "electrical_conductivity_25c_in_uS/cm",
    "pH",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L",
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "Li_in_mmol/L",
    "K_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "F_in_umol/L",
    "NO3_in_mmol/L",
    "H2SiO3_in_umol/L",
    "HS_in_mmol/L",
    "Database_number",
]

# -------------------------- Mapping: Quelle -> Ziel
# Beachten: Spaltennamen immer KLEINSCHREIBEN, da vorhergehend einheitliche Konvertierung in Kleinbuchstaben und case sensitive
Sheet1 = {
    "county": "location",
    "wellname": "well_or_spring_name",
    "latitude": "WGS84_latitude",
    "longitude": "WGS84_longitude",
    "depthlower": "depth_bgl_in_m",
    "lithology": "rock_type",
    "period": "stratigraphic_period",
    "temp": "temperature_in_c",
    "ph": "pH",
    "tds": "total_dissolved_solids_in_mg/L",   # <-- erst mg/L, später korrekt nach mmol/L durhc Addition 
    "na": "Na_in_mmol/L",
    "mg": "Mg_in_mmol/L",
    "ca": "Ca_in_mmol/L",
    "cl": "Cl_in_mmol/L",
    "so4": "SO4_in_mmol/L",
    "hco3": "HCO3_in_mmol/L",
    "li": "Li_in_mmol/L",
    "k": "K_in_mmol/L",
    "sr": "Sr_in_umol/L",
    "nh4": "NH4_in_umol/L",
    "fetot": "Fe_in_mmol/L",
    "mn": "Mn_in_mmol/L",
    "f": "F_in_umol/L",
    "no3": "NO3_in_mmol/L",
    "hs": "HS_in_mmol/L",
    "database_number": "Database_number",
}

<div class="alert alert-info">
    Die Hilfsfunktionen<br>
    -> _normalize_columns<br>
    -> _available_mapping<br>
    -> read_and_map<br>
    wurden bereits erzeugt und können einfach erneut abgerufen werden
</div>

In [206]:
# -------------------------------- Dateien/Ordner --------------------------------
# mid_usgs_path
# usgs_path

# ---------------------------------- Excel öffnen --------------------------------
xls = pd.ExcelFile(usgs_path)


# --------------------------- Einlesen + Umbenennen je Blatt ---------------------
df_sheet1 = read_and_map(xls, SHEET_1, Sheet1)
# in diesem Fall gibt es nur ein Blatt -> daher kann df_combined einfach auf df_sheet1 gesetzt werden
df_combined = df_sheet1.copy()

# --------------- Doppelte Spaltennamen entfernen ----------------------------------
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()].copy()



# ----------------- Fehlende Zielspalten ergänzen -----------------------------------
for col in TARGET_COLUMNS:
    if col not in df_combined.columns:
        df_combined[col] = pd.NA


# --------------------- Auf Zielstruktur bringen ------------------------------------
df_out = df_combined[TARGET_COLUMNS].copy()


# ------------------------------ Export ---------------------------------------------
out_usgs_path = mid_usgs_path / "before_conversion_usgs_mapped.csv"
df_out.to_csv(out_usgs_path, index=False, encoding="utf-8-sig")


print(f"USGS-Mapping abgeschlossen: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"CSV-Pfad: {out_usgs_path.resolve()}")

USGS-Mapping abgeschlossen: 114943 Zeilen, 30 Spalten
CSV-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank\before_conversion_usgs_mapped.csv


<div class="alert alert-info">
    <b>Schritt 2.2 | Unwandlung in Excel</b><br>
    Um sich die neu entstandende Datei in Form einer Excel-Tabelle anschauen zu können findet eine Konvertierung der .csv zu .xlsx statt<br><br>
    Dies wird als Funktion verankert, sodass diese immer wieder nach Veränderung der .csv ausgeführt werden kann, um eine Darstellung via Excel zu ermöglichen.
</div>

In [207]:
# ---------------------- Dateipfade für Zwischenstände ----------------------
csv_path = mid_usgs_path / "before_conversion_usgs_mapped.csv"
excel_path = mid_usgs_path / "before_conversion_usgs_mapped_readable.xlsx"

print(f"CSV-Pfad:   {csv_path.resolve()}")
print(f"Excel-Pfad: {excel_path.resolve()}")

CSV-Pfad:   C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank\before_conversion_usgs_mapped.csv
Excel-Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank\before_conversion_usgs_mapped_readable.xlsx


In [208]:
# csv_to_excel(csv_path, excel_path)

<div class="alert alert-info">
    <b>Schritt 3 | Umwandlung zu einheitlichem NaN Datentyp</b><br>
    "Leere Elemente" oder solche die mit "N/D" gekennzeichnet sind werde zum Pandas Typ NaN umgewandelt
</div>

In [209]:
df_out = df_out.replace(r'^\s*$|N/D', np.nan, regex=True)
df_out.to_csv(out_usgs_path, index=False, encoding="utf-8-sig")

In [210]:
csv_to_excel(csv_path, excel_path)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank\before_conversion_usgs_mapped_readable.xlsx


<div class="alert alert-info">
    <b>Schritt 4 | Umwandlung der Einheiten</b><br>
    Da die Werte noch nicht in ihrer korrekten Einheit vorliegen (bspw. mmol/L oder umol/L), sondern in ihrer Ausgangsform (oft mg/L) muss im nächsten Schritt genau dies getan werden.
</div>

In [211]:
# --------------------------- Zielspalten ----------------------------
TARGET_COLUMNS = [
    "location","well_or_spring_name","WGS84_latitude","WGS84_longitude","depth_bgl_in_m","rock_type","stratigraphic_period",
    "temperature_in_c","electrical_conductivity_25c_in_uS/cm","pH","redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",
    "O2_in_mmol/L","Na_in_mmol/L","Mg_in_mmol/L","Ca_in_mmol/L","Cl_in_mmol/L","SO4_in_mmol/L",
    "HCO3_in_mmol/L","Li_in_mmol/L","K_in_mmol/L","Sr_in_umol/L","NH4_in_umol/L","Fe_in_mmol/L",
    "Mn_in_mmol/L","F_in_umol/L","NO3_in_mmol/L","H2SiO3_in_umol/L","HS_in_mmol/L", "Database_number",
]

# --------------------------------------- Molmassen --------------------------------------
M = {
    "O2": 31.998, "Na": 22.989769, "Mg": 24.305, "Ca": 40.078, "Cl": 35.45, "SO4": 96.06,
    "HCO3": 61.0168, "Li": 6.94, "K": 39.0983, "Sr": 87.62, "NH4": 18.038, "Fe": 55.845,
    "Mn": 54.938, "F": 18.998, "NO3": 62.0049, "H2SiO3": 78.11, "HS": 33.07,
}

# Einlesen
df = pd.read_csv(csv_path)

# -------------------------------- Sicherstellen ----------------------------------------
for c in TARGET_COLUMNS:
    if c not in df.columns:
        df[c] = np.nan


In [212]:
# Umrechnungen (Annahme: Spalten mit *_in_mmol/L / *_in_umol/L enthalten noch mg/L)

# ------- mg/L -> mmol/L
for col, sp in mmol_map.items():
    if col in df.columns:
        df[col] = mgL_to_mmolL(df[col], M[sp])

# ------- mg/L -> µmol/L
for col, sp in umol_map.items():
    if col in df.columns:
        df[col] = mgL_to_umolL(df[col], M[sp])


# --------------------- Negative Ausreißer durch Mess-/Importfehler neutralisieren ---------------------------
chem_cols = list(mmol_map.keys()) + list(umol_map.keys())
df[chem_cols] = df[chem_cols].where(df[chem_cols] >= 0, np.nan)


# -------------------------- fehlende Spalten auffüllen (falls im CSV nicht vorhanden) -----------------------------
for c in tds_mmols_cols + tds_umols_cols:
    if c not in df.columns:
        df[c] = np.nan

tds_mmol_part = df[tds_mmols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
tds_umol_part = df[tds_umols_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1) / 1000.0
df["total_dissolved_solids_in_mmol/L"] = tds_mmol_part + tds_umol_part


# --------------------------------------------- Ausgabe in Zielstruktur --------------------------------------------
after_conversion_usgs_path = mid_usgs_path / "after_conversion_usgs_mapped.csv"
df_out = df[TARGET_COLUMNS].copy()
after_conversion_usgs_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(after_conversion_usgs_path, index=False, encoding="utf-8-sig")

print(f"Einheiten-Umrechnung + TDS-Neuberechnung: {len(df_out)} Zeilen, {df_out.shape[1]} Spalten")
print(f"Gespeichert unter Pfad: {after_conversion_usgs_path.resolve()}")

Einheiten-Umrechnung + TDS-Neuberechnung: 114943 Zeilen, 30 Spalten
Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank\after_conversion_usgs_mapped.csv


<div class="alert alert-info">
    <b>Schritt 5 | Ausgabe der fertig aufbereiteten Dateien</b><br>
    Die fertig aufbereitenden Dateien werden der übersichts Halber einmal im dedizierten Zwischenstands-Ordner gespeichert und in den Ordner für "Angepasste Datenbanken" gespeichert
</div>

In [213]:
# ---------------------- Dateipfade für konvertierte REFLECT-Datenbank ----------------------
csv_path = mid_usgs_path / "after_conversion_usgs_mapped.csv"
excel_path = mid_usgs_path / "after_conversion_usgs_mapped_readable.xlsx"

In [214]:
# --------------------------- CSV -> Excel konvertieren ---------------------------
csv_to_excel(csv_path, excel_path)

# --------------------------- Ausgabeordner definieren ---------------------------
output_path = base_dir / "Angepasste_Datenbanken"
output_path.mkdir(parents=True, exist_ok=True)

Excel-Pfad:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Zwischenstände\USGS_Datenbank\after_conversion_usgs_mapped_readable.xlsx


In [215]:
# --------------------------- Finale, bereinigte REFLECT-Datenbank speichern --------------------------- 
usgs_output_path = output_path / "USGS_cleaned_Database.csv" 
usgs_output_path.parent.mkdir(parents=True, exist_ok=True) 

df_out.to_csv(usgs_output_path, index=False, encoding="utf-8-sig") 
print(f"Gespeichert unter Pfad: {usgs_output_path.resolve()}")

Gespeichert unter Pfad: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\USGS_cleaned_Database.csv


<div class="alert alert-info">
    <b>Zusatz | Überprüfung geographischer Lage</b><br>
    In folgender Darstellung kann überprüft werden, ob die Datenpunkte tatsächlich am eigentlichen Ort liegen - oder ob hier eine falsche Darstellung der Koordinaten verwendet werden muss - und diese anschließend erneut umgerechnet werden muss
</div>

In [216]:
map_path = base_dir / "Kartierung"
usgs_map_path = map_path / "USGS_Map.html" 

In [217]:
from folium.plugins import FastMarkerCluster

df = df_out.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# ------------------------------ Kartenmittelpunkt setzen -----------------------------------
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# ---------------------- FastMarkerCluster mit Koordinaten füllen ---------------------------
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(usgs_map_path)

<div class="alert alert-danger">
    <h1><b>Zusammenfügen der Datenbanken</b></h1>
</div>

In [218]:
def merge_csv_folder(output_path) -> Path:

    base = Path(output_path)
    if not base.exists():
        raise FileNotFoundError(f"Ordner existiert nicht: {base.resolve()}")

    csv_files = sorted(p for p in base.glob("*.csv") if p.is_file())
    if not csv_files:
        raise FileNotFoundError(f"Keine .csv-Dateien gefunden in: {base.resolve()}")

    print("Gefundene CSV-Dateien:")
    for p in csv_files:
        print(f"  - {p.name}")

    # -------------------------------- Header der ersten Datei als Referenz ------------------------
    ref_cols = pd.read_csv(csv_files[0], nrows=0).columns.tolist()

    # --------------------------- prüfen, dass alle Header exakt übereinstimmen -------------------
    for p in csv_files[1:]:
        cols = pd.read_csv(p, nrows=0).columns.tolist()
        if cols != ref_cols:
            raise ValueError(
                f"Spaltenkopf unterscheidet sich in Datei: {p.name}\n"
                f"Erwartet: {ref_cols}\n"
                f"Gefunden: {cols}"
            )

    # ----------------------------------------- Einlesen & zusammenfügen --------------------------
    parts = []
    total_rows = 0
    print("\nZeilenanzahl (ohne Header) je Datei:")
    for p in csv_files:
        df_part = pd.read_csv(p)

        parts.append(df_part)
        rows = len(df_part)
        total_rows += rows
        print(f"  - {p.name}: {rows}")

    merged = pd.concat(parts, axis=0, ignore_index=True)

    print(f"\nGesamtzeilen (neu erstellte Datenbank): {len(merged)}")
    assert len(merged) == total_rows, "Summierte Zeilen ≠ Anzahl Zeilen im Merge (unerwartet)."

    # -------------------------------- Zielordner mit Zeitstempel erstellen ----------------------
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")  # YYYY-MM-DD-HH-MM-SS
    target_dir = base / "Komplette_Datenbank" / timestamp
    
    target_dir.mkdir(parents=True, exist_ok=True)
    out_path = target_dir / "Komplette_Datenbank.csv"
    merged.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"\nusammenführung abgeschlossen.")
    print(f"Ausgabedatei: {out_path.resolve()}")

    return out_path

# ------------------------------------------- Aufruf ----------------------------------------------
merged_path = merge_csv_folder(output_path)
merged = pd.read_csv(merged_path)

Gefundene CSV-Dateien:
  - ALPS_cleaned_Database.csv
  - ARGONNE_cleaned_Database.csv
  - AUSTRALIA-MATARANKA_cleaned_Database.csv
  - CAES2014_cleaned_Database.csv
  - GANDAKI_cleaned_Database.csv
  - HESSEN_cleaned_Database.csv
  - MDPI_Tibet_cleaned_Database.csv
  - REFLECT_cleaned_Database.csv
  - SVZ-CHILE_cleaned_Database.csv
  - THAILAND_cleaned_Database.csv
  - USGS_cleaned_Database.csv
  - XIANSHUIHE_cleaned_Database.csv
  - YUKON_cleaned_Database.csv

Zeilenanzahl (ohne Header) je Datei:
  - ALPS_cleaned_Database.csv: 311
  - ARGONNE_cleaned_Database.csv: 51078
  - AUSTRALIA-MATARANKA_cleaned_Database.csv: 84
  - CAES2014_cleaned_Database.csv: 42
  - GANDAKI_cleaned_Database.csv: 12
  - HESSEN_cleaned_Database.csv: 1035
  - MDPI_Tibet_cleaned_Database.csv: 52


C:\Users\lucca\AppData\Local\Temp\ipykernel_23256\109384912.py:33: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_part = pd.read_csv(p)


  - REFLECT_cleaned_Database.csv: 5105
  - SVZ-CHILE_cleaned_Database.csv: 30
  - THAILAND_cleaned_Database.csv: 31
  - USGS_cleaned_Database.csv: 114943
  - XIANSHUIHE_cleaned_Database.csv: 280
  - YUKON_cleaned_Database.csv: 88

Gesamtzeilen (neu erstellte Datenbank): 173091

usammenführung abgeschlossen.
Ausgabedatei: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Komplette_Datenbank\2025-11-23_11-44-33\Komplette_Datenbank.csv


C:\Users\lucca\AppData\Local\Temp\ipykernel_23256\109384912.py:60: DtypeWarning: Columns (4,5,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  merged = pd.read_csv(merged_path)


In [219]:
map_path = base_dir / "Kartierung" / "Fertige_Karte"
merged_map_path = map_path / "Merged_Map.html" 

In [220]:
df = merged.copy()
cols = ["WGS84_latitude", "WGS84_longitude", "well_or_spring_name"]
pts = df[cols].dropna(subset=["WGS84_latitude", "WGS84_longitude"])

# Kartenmittelpunkt setzen
center = [pts["WGS84_latitude"].mean(), pts["WGS84_longitude"].mean()]
m = folium.Map(location=center, zoom_start=6, tiles="OpenStreetMap", control_scale=True)

# FastMarkerCluster mit Koordinaten füllen
coords = pts[["WGS84_latitude", "WGS84_longitude"]].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

folium.plugins.Fullscreen().add_to(m)
folium.LayerControl().add_to(m)

m.save(merged_map_path)